### 1. Load Dependencies

In [6]:
%pip install nltk


Note: you may need to restart the kernel to use updated packages.


In [7]:
import nltk

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to /Users/sushma/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sushma/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/sushma/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/sushma/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [8]:
%pip install pandas numpy scikit-learn nltk sentence-transformers streamlit pdfplumber PyPDF2 matplotlib joblib

Note: you may need to restart the kernel to use updated packages.


### 2.Import Dataset and Explore

In [9]:
import pandas as pd
import sklearn
from sentence_transformers import SentenceTransformer

print("Setup successful!")

Setup successful!


In [10]:
df = pd.read_csv('/Users/sushma/Documents/SRH Class Notes/Semester 2/Natural Language Processing/NLP Final Project/NLP-CareerCompass/postings.csv')

print(df.shape)
df.head()

(123849, 31)


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,This position requires a baseline understandin...,1.712896e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,1.713452e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     23320 non-null   float64
 13  original_liste

In [12]:
useful_cols = [
    "job_id",
    "company_name",
    "title",
    "description",
    "location",
    "formatted_work_type",
    "formatted_experience_level",
    "skills_desc",
    "work_type",
    "normalized_salary",
    "job_posting_url"
]

df = df[useful_cols]

df.head()

,job_id,company_name,title,description,location,formatted_work_type,formatted_experience_level,skills_desc,work_type,normalized_salary,job_posting_url
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,"Princeton, NJ",Full-time,NaN,Requirements: \n\nWe are seeking a College or ...,FULL_TIME,38480.0,https://www.linkedin.com/jobs/view/921716/?trk...
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...","Fort Collins, CO",Full-time,NaN,NaN,FULL_TIME,83200.0,https://www.linkedin.com/jobs/view/1829192/?tr...
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,"Cincinnati, OH",Full-time,NaN,We are currently accepting resumes for FOH - A...,FULL_TIME,55000.0,https://www.linkedin.com/jobs/view/10998357/?t...
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,"New Hyde Park, NY",Full-time,NaN,This position requires a baseline understandin...,FULL_TIME,157500.0,https://www.linkedin.com/jobs/view/23221523/?t...
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,"Burlington, IA",Full-time,NaN,NaN,FULL_TIME,70000.0,https://www.linkedin.com/jobs/view/35982263/?t...


In [13]:
df = df.drop_duplicates(subset=["title", "description"])

df = df.dropna(subset=["title", "description"])

print(df.shape)

(110898, 11)


In [14]:
df = df.dropna(
    subset=[
        "title",
        "description"
    ]
)

print("After dropping empty title/description:", df.shape)

After dropping empty title/description: (110898, 11)


In [15]:
df["company_name"] = df["company_name"].fillna("Unknown")

df["formatted_experience_level"] = (
    df["formatted_experience_level"]
    .fillna("Not Specified")
)

df["normalized_salary"] = (
    df["normalized_salary"]
    .fillna(-1)
)

In [16]:
df = df.drop(columns=["skills_desc"])

In [17]:
df["job_text"] = (
    df["title"].astype(str)
    + " "
    + df["description"].astype(str)
)

In [18]:
print(df["job_text"].isnull().sum())

0


In [19]:
df = df.sample(
    n=10000,
    random_state=42
).reset_index(drop=True)

In [20]:
print(df.shape)

(10000, 11)


In [21]:
df.to_csv(
    "linkedin_jobs_clean.csv",
    index=False
)

print("Saved linkedin_jobs_clean.csv")

Saved linkedin_jobs_clean.csv


In [22]:
df = pd.read_csv("linkedin_jobs_clean.csv")
df.head()

,job_id,company_name,title,description,location,formatted_work_type,formatted_experience_level,work_type,normalized_salary,job_posting_url,job_text
0,3884896598,TravelNurseSource,"Travel Nurse RN - ICU/Critical Care - $2,016 p...",TravelNurseSource is working with Amare Medica...,"Hickory, NC",Full-time,Mid-Senior level,FULL_TIME,-1.0,https://www.linkedin.com/jobs/view/3884896598/...,"Travel Nurse RN - ICU/Critical Care - $2,016 p..."
1,3889748025,Liberty Healthcare and Rehabilitation Services,PHYSICAL THERAPIST (PT) - LIBERTY COMMONS OF H...,\nLiberty Cares\nWith Compassion\nAt Liberty H...,"Weldon, NC",Full-time,Not Specified,FULL_TIME,-1.0,https://www.linkedin.com/jobs/view/3889748025/...,PHYSICAL THERAPIST (PT) - LIBERTY COMMONS OF H...
2,3887900308,Tradesmen International,Carpenter,Description\n\nTradesmen International is imme...,"Oak Park, IL",Full-time,Entry level,FULL_TIME,-1.0,https://www.linkedin.com/jobs/view/3887900308/...,Carpenter Description\n\nTradesmen Internation...
3,3887853674,Adaptiva,Technical Writer,"Technical WriterAbout Adaptiva\nAdaptiva, the ...","Kirkland, WA",Full-time,Mid-Senior level,FULL_TIME,92500.0,https://www.linkedin.com/jobs/view/3887853674/...,Technical Writer Technical WriterAbout Adaptiv...
4,3904515350,Marcum Search,Senior Billing Administrator,"Marcum Search is working with a high profile, ...",Los Angeles Metropolitan Area,Full-time,Mid-Senior level,FULL_TIME,-1.0,https://www.linkedin.com/jobs/view/3904515350/...,Senior Billing Administrator Marcum Search is...


### 3. Preprocessing

In [23]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [24]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to /Users/sushma/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/sushma/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sushma/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/sushma/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/sushma/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [25]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [26]:
#preprocess
def preprocess_text(text):
    # lowercase
    text = str(text).lower()

    # remove urls
    text = re.sub(r"http\S+|www\S+", "", text)

    # remove punctuation and numbers
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # tokenize
    tokens = word_tokenize(text)

    # remove stopwords and short words
    tokens = [
        word for word in tokens
        if word not in stop_words and len(word) > 2
    ]

    # lemmatize
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens)

In [27]:
df["clean_job_text"] = df["job_text"].apply(preprocess_text)

In [28]:
print("ORIGINAL:")
print(df.loc[0, "job_text"])

print("\n" + "="*80 + "\n")

print("CLEANED:")
print(df.loc[0, "clean_job_text"])

ORIGINAL:
Travel Nurse RN - ICU/Critical Care - $2,016 per week in Hickory, NC TravelNurseSource is working with Amare Medical Network to find a qualified ICU/Critical Care RN in Hickory, North Carolina, 28602!

Pay Information

$2,016 per week

About The Position

ICU registered nurses (RNs) provide care to critical medical, surgical and traumatized patients. Close monitoring and observation of the patient's heart rate, blood pressure and respiration for signs of distress represent a primary function of and ICU nurse. They also educate patients and their families while providing emotional support.

Job Requirements

Required for Onboarding

General ICU RNICU - Skills

About Amare Medical Network

At Amare Medical Network, we specialize in matching nursing professionals with jobs they love. We understand you want more flexibility, better benefits, higher pay, and a supportive working environment where your skills are valued in exactly the way you deserve.

Our purpose is to care for th

In [29]:
df = df[
    df["clean_job_text"].str.strip() != ""
].reset_index(drop=True)

print(df.shape)

(10000, 12)


In [30]:

df.to_csv(
    "linkedin_jobs_preprocessed.csv",
    index=False
)
print("Saved linkedin_jobs_preprocessed.csv")


Saved linkedin_jobs_preprocessed.csv


### 4. TF-IDF Baseline recommender

In [31]:
# Load preprocessed data
import pandas as pd

df = pd.read_csv("linkedin_jobs_preprocessed.csv")

print(df.shape)
df[["title", "clean_job_text"]].head()

(10000, 12)


,title,clean_job_text
0,"Travel Nurse RN - ICU/Critical Care - $2,016 p...",travel nurse icu critical care per week hickor...
1,PHYSICAL THERAPIST (PT) - LIBERTY COMMONS OF H...,physical therapist liberty common halifax coun...
2,Carpenter,carpenter description tradesman international ...
3,Technical Writer,technical writer technical writerabout adaptiv...
4,Senior Billing Administrator,senior billing administrator marcum search wor...


In [32]:
# 2. Create tf-idf matrix
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df["clean_job_text"])

print(tfidf_matrix.shape)

(10000, 5000)


In [33]:
# 3. Create resume preprocessing
resume_text = """
Experienced data analyst skilled in Python, SQL, Excel, Tableau,
Power BI, data visualization, reporting, and machine learning.
"""

In [34]:
clean_resume = preprocess_text(resume_text)
print(clean_resume)

experienced data analyst skilled python sql excel tableau power data visualization reporting machine learning


In [35]:
# 4. Transform resume using same TF-IDF vectorizer
resume_vector = tfidf_vectorizer.transform([clean_resume])

print(resume_vector.shape)

(1, 5000)


Compute cosine similarity

In [36]:
# 5. Compute cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

tfidf_scores = cosine_similarity(resume_vector, tfidf_matrix).flatten()

print(tfidf_scores[:10])

[0.         0.01560323 0.01649476 0.00762971 0.         0.
 0.0142131  0.         0.10733541 0.02086178]


In [37]:
# 6. Get top 10 jobs
top_indices = tfidf_scores.argsort()[::-1][:10]

tfidf_recommendations = df.iloc[top_indices].copy()
tfidf_recommendations["tfidf_score"] = tfidf_scores[top_indices]

tfidf_recommendations[
    ["title", "company_name", "location", "formatted_experience_level", "tfidf_score"]
]

,title,company_name,location,formatted_experience_level,tfidf_score
2614,Lead Machine Learning Engineer,Confidential Jobs,United States,Mid-Senior level,0.487454
1834,Palantir Developer,Tata Consultancy Services,"Cleveland, OH",Entry level,0.479114
8582,Technical Lead,CodeMonk,United States,Mid-Senior level,0.420952
9425,Senior Machine Learning Engineer,Harnham,"Los Angeles, CA",Mid-Senior level,0.411580
2376,Business Data Analyst,Catapult Solutions Group,"San Diego, CA",Mid-Senior level,0.410448
3504,Machine Learning Engineer,Cloud Resources LLC,"New York, NY",Not Specified,0.370389
2394,Business Data Analyst,Piper Companies,"McLean, VA",Mid-Senior level,0.368059
2378,Business Analyst III - Tableau (Remote),Talentify.io,United States,Mid-Senior level,0.363386
5499,Power BI Developer,Optomi,United States,Mid-Senior level,0.360739
3740,Data Analyst - Power BI & SQL,ZealTech,"San Francisco, CA",Not Specified,0.357396


### 5a. SENTENCE BERT EMBEDDINGS

In [38]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [39]:
import pandas as pd

df = pd.read_csv("linkedin_jobs_preprocessed.csv")

print(df.shape)

(10000, 12)


In [40]:
# 3 load sbert model
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [41]:
# create job embeddings
job_texts = df["job_text"].fillna("").tolist()

job_embeddings = sbert_model.encode(
    job_texts,
    show_progress_bar=True,
    batch_size=32
)

print(job_embeddings.shape)

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

(10000, 384)


In [42]:
# 5. Save embeddings
np.save("job_sbert_embeddings.npy", job_embeddings)

print("Saved job_sbert_embeddings.npy")

Saved job_sbert_embeddings.npy


In [43]:
resume_text = """
Experienced data analyst skilled in Python, SQL, Excel, Tableau,
Power BI, data visualization, reporting, dashboards, and machine learning.
"""

In [44]:
resume_embedding = sbert_model.encode([resume_text])
print(resume_embedding.shape)

(1, 384)


In [45]:
# Compute SBERT similarity
sbert_scores = cosine_similarity(
    resume_embedding,
    job_embeddings
).flatten()

print(sbert_scores[:10])

[0.1881183  0.08655276 0.21693562 0.24265377 0.29135677 0.25492975
 0.1668716  0.21699028 0.2412162  0.19180693]


In [46]:

top_indices = sbert_scores.argsort()[::-1][:10]

sbert_recommendations = df.iloc[top_indices].copy()
sbert_recommendations["sbert_score"] = sbert_scores[top_indices]

sbert_recommendations[
    ["title", "company_name", "location", "formatted_experience_level", "sbert_score"]
]

,title,company_name,location,formatted_experience_level,sbert_score
2378,Business Analyst III - Tableau (Remote),Talentify.io,United States,Mid-Senior level,0.706202
1310,Data Scientist,"V-Soft Consulting Group, Inc.","Manhattan, NY",Entry level,0.673356
6469,Data Analyst,USCTEK - Recruiting & Staffing,United States,Not Specified,0.672319
4160,Data Analyst,SNVA Technologies,"Juno Beach, FL",Not Specified,0.665626
4670,Data Engineer (LEAD),RICEFW Technologies Inc,"Allentown, PA",Mid-Senior level,0.664252
9477,Data Analyst,Lume Consulting Group,Greater Sacramento,Not Specified,0.663480
8668,Data Analyst,"LanceSoft, Inc.","Richmond, VA",Mid-Senior level,0.663034
4391,Credit Data Analyst 2 (Remote),Nordstrom,United States,Mid-Senior level,0.652438
3626,Data Science Intern DIN47,Data Glacier,United States,Not Specified,0.642379
2394,Business Data Analyst,Piper Companies,"McLean, VA",Mid-Senior level,0.640904


## Fine Tune SBERT

In [47]:
#step 5 fine tune sbert
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import pandas as pd
import random

/var/folders/8v/frb1qkxn6tvb4c7lz7kfr90r0000gn/T/ipykernel_27010/3637533002.py:2: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


In [48]:
df = pd.read_csv("linkedin_jobs_preprocessed.csv")
df = df.dropna(subset=["job_text"]).reset_index(drop=True)

In [49]:
def create_augmented_text(text):
    words = str(text).split()

    if len(words) <= 30:
        return str(text)

    keep_count = int(len(words) * 0.75)
    selected_words = random.sample(words, keep_count)

    return " ".join(selected_words)

In [50]:
train_df = df.sample(n=3000, random_state=42)

train_examples = []

for text in train_df["job_text"]:
    original = str(text)
    augmented = create_augmented_text(original)

    train_examples.append(
        InputExample(texts=[original, augmented])
    )

print(len(train_examples))

3000


In [51]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [52]:
train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=16
)

train_loss = losses.MultipleNegativesRankingLoss(model)

In [53]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=100,
    show_progress_bar=True
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


In [54]:
model.save("career_sbert_finetuned")

print("Fine-tuned SBERT saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned SBERT saved.


### 5c. Generate fine tuned embeddings


In [55]:
from sentence_transformers import SentenceTransformer
import numpy as np

sbert_model = SentenceTransformer("career_sbert_finetuned")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [56]:
job_texts = df["job_text"].fillna("").tolist()

job_embeddings_finetuned = sbert_model.encode(
    job_texts,
    show_progress_bar=True,
    batch_size=32
)

print(job_embeddings_finetuned.shape)

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

(10000, 384)


In [57]:
np.save("job_sbert_finetuned_embeddings.npy", job_embeddings_finetuned)

print("Saved fine-tuned job embeddings.")

Saved fine-tuned job embeddings.


## K MEANS CLUSTERING

In [58]:
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [59]:
job_embeddings_norm = normalize(job_embeddings_finetuned)

In [60]:
silhouette_scores = {}

for k in range(5, 16):
    kmeans_temp = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels_temp = kmeans_temp.fit_predict(job_embeddings_norm)

    score = silhouette_score(
        job_embeddings_norm,
        labels_temp,
        sample_size=2000,
        random_state=42
    )

    silhouette_scores[k] = score
    print(f"K={k}, Silhouette Score={score:.4f}")

K=5, Silhouette Score=0.0387
K=6, Silhouette Score=0.0349
K=7, Silhouette Score=0.0357
K=8, Silhouette Score=0.0342
K=9, Silhouette Score=0.0305
K=10, Silhouette Score=0.0296
K=11, Silhouette Score=0.0285
K=12, Silhouette Score=0.0290
K=13, Silhouette Score=0.0277
K=14, Silhouette Score=0.0267
K=15, Silhouette Score=0.0294


In [61]:
best_k = 8   # change this based on best silhouette score

kmeans_model = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans_model.fit_predict(job_embeddings_norm)

In [62]:
df["cluster"].value_counts().sort_index()

cluster
0    1413
1    1456
2    1806
3     842
4     663
5     986
6    1362
7    1472
Name: count, dtype: int64

In [63]:
for cluster_id in sorted(df["cluster"].unique()):
    print(f"\nCluster {cluster_id}")
    print("-" * 50)
    print(
        df[df["cluster"] == cluster_id]["title"]
        .value_counts()
        .head(10)
    )


Cluster 0
--------------------------------------------------
title
Administrative Assistant           15
Receptionist                       14
General Manager                     9
Executive Assistant                 8
Associate Attorney                  6
Human Resources Generalist          6
Human Resources Manager             6
Property Manager                    5
Office Manager                      5
Customer Service Representative     5
Name: count, dtype: int64

Cluster 1
--------------------------------------------------
title
Software Engineer                          11
Data Analyst                               10
Network Engineer                           10
Business Analyst                            9
Senior Software Engineer                    8
Data Engineer                               8
Board Certified Behavior Analyst (BCBA)     7
Python Developer                            6
System Engineer                             6
Senior Data Engineer                        

In [64]:
import joblib

df.to_csv("linkedin_jobs_clustered.csv", index=False)
joblib.dump(kmeans_model, "kmeans_model.pkl")

print("Saved clustered dataset and K-Means model.")

Saved clustered dataset and K-Means model.


In [65]:
# Number of jobs in each cluster
print("Jobs per cluster:")
print(df["cluster"].value_counts().sort_index())

print("\n" + "="*80)

# Print the most common job titles in each cluster
for cluster_id in sorted(df["cluster"].unique()):

    print(f"\nCLUSTER {cluster_id}")
    print("-" * 50)

    top_titles = (
        df[df["cluster"] == cluster_id]["title"]
        .value_counts()
        .head(15)
    )

    print(top_titles)

    print("\n" + "="*80)

Jobs per cluster:
cluster
0    1413
1    1456
2    1806
3     842
4     663
5     986
6    1362
7    1472
Name: count, dtype: int64


CLUSTER 0
--------------------------------------------------
title
Administrative Assistant           15
Receptionist                       14
General Manager                     9
Executive Assistant                 8
Associate Attorney                  6
Human Resources Generalist          6
Human Resources Manager             6
Property Manager                    5
Office Manager                      5
Customer Service Representative     5
Associate Team Leader               4
Front Desk Agent                    4
Janitor                             4
Service Leader                      4
Community Manager                   4
Name: count, dtype: int64


CLUSTER 1
--------------------------------------------------
title
Software Engineer                          11
Data Analyst                               10
Network Engineer                          

# HYBRID MODEL 
## TF-IDF baseline + fine-tuned SBERT embeddings + K-Means cluster filtering

In [66]:
import pandas as pd
import numpy as np
import joblib

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

In [67]:
# STEP 7a load everything
df = pd.read_csv("linkedin_jobs_clustered.csv")

sbert_model = SentenceTransformer("career_sbert_finetuned")
kmeans_model = joblib.load("kmeans_model.pkl")

job_embeddings = np.load("job_sbert_finetuned_embeddings.npy")
job_embeddings_norm = normalize(job_embeddings)

print(df.shape)
print(job_embeddings_norm.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(10000, 13)
(10000, 384)


In [68]:
#rebuild tfidf vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df["clean_job_text"])

In [69]:
#Hybrid recommendation function
def recommend_jobs_hybrid(resume_text, top_n=10):
    # 1. Preprocess resume for TF-IDF
    clean_resume = preprocess_text(resume_text)

    # 2. TF-IDF similarity
    resume_tfidf = tfidf_vectorizer.transform([clean_resume])
    tfidf_scores = cosine_similarity(resume_tfidf, tfidf_matrix).flatten()

    # 3. Fine-tuned SBERT similarity
    resume_embedding = sbert_model.encode([resume_text])
    resume_embedding_norm = normalize(resume_embedding)

    sbert_scores = cosine_similarity(
        resume_embedding_norm,
        job_embeddings_norm
    ).flatten()

    # 4. Find nearest cluster
    resume_cluster = kmeans_model.predict(resume_embedding_norm)[0]

    # 5. Keep only jobs from same cluster
    cluster_indices = df[df["cluster"] == resume_cluster].index

    # 6. Hybrid score
    hybrid_scores = (0.4 * tfidf_scores) + (0.6 * sbert_scores)

    # 7. Rank only inside cluster
    cluster_scores = hybrid_scores[cluster_indices]

    top_cluster_positions = cluster_scores.argsort()[::-1][:top_n]
    top_indices = cluster_indices[top_cluster_positions]

    recommendations = df.loc[top_indices].copy()
    recommendations["tfidf_score"] = tfidf_scores[top_indices]
    recommendations["sbert_score"] = sbert_scores[top_indices]
    recommendations["hybrid_score"] = hybrid_scores[top_indices]
    recommendations["resume_cluster"] = resume_cluster

    return recommendations[
        [
            "title",
            "company_name",
            "location",
            "formatted_work_type",
            "formatted_experience_level",
            "normalized_salary",
            "tfidf_score",
            "sbert_score",
            "hybrid_score",
            "cluster",
            "job_posting_url"
        ]
    ]

In [70]:
resume_text = """
Data analyst with experience in Python, SQL, Excel, Tableau, Power BI,
data visualization, dashboard development, reporting, and machine learning.
"""

recommend_jobs_hybrid(resume_text, top_n=10)

,title,company_name,location,formatted_work_type,formatted_experience_level,normalized_salary,tfidf_score,sbert_score,hybrid_score,cluster,job_posting_url
2378,Business Analyst III - Tableau (Remote),Talentify.io,United States,Full-time,Mid-Senior level,150280.0,0.415956,0.704653,0.589174,1,https://www.linkedin.com/jobs/view/3905658764/...
1913,Power BI Developer,Odyssey Information Services,"Piqua, OH",Full-time,Mid-Senior level,-1.0,0.434300,0.597430,0.532178,1,https://www.linkedin.com/jobs/view/3887102526/...
1834,Palantir Developer,Tata Consultancy Services,"Cleveland, OH",Full-time,Entry level,-1.0,0.523194,0.485253,0.500429,1,https://www.linkedin.com/jobs/view/3885815054/...
5499,Power BI Developer,Optomi,United States,Contract,Mid-Senior level,133120.0,0.422103,0.544668,0.495642,1,https://www.linkedin.com/jobs/view/3887100812/...
3740,Data Analyst - Power BI & SQL,ZealTech,"San Francisco, CA",Contract,Not Specified,-1.0,0.423915,0.537764,0.492224,1,https://www.linkedin.com/jobs/view/3885113214/...
7986,Business Intelligence Analyst,Emergere Technologies LLC.,"Dallas, TX",Contract,Not Specified,-1.0,0.330533,0.592595,0.487770,1,https://www.linkedin.com/jobs/view/3900964322/...
2376,Business Data Analyst,Catapult Solutions Group,"San Diego, CA",Full-time,Mid-Senior level,133712.8,0.446753,0.500396,0.478939,1,https://www.linkedin.com/jobs/view/3901389257/...
4160,Data Analyst,SNVA Technologies,"Juno Beach, FL",Contract,Not Specified,-1.0,0.290221,0.576056,0.461722,1,https://www.linkedin.com/jobs/view/3901373790/...
8668,Data Analyst,"LanceSoft, Inc.","Richmond, VA",Contract,Mid-Senior level,-1.0,0.300303,0.547829,0.448819,1,https://www.linkedin.com/jobs/view/3899534432/...
2629,Power BI Developer & Reporting Analyst II,Akkodis,"Houston, TX",Contract,Associate,73455.2,0.264394,0.565339,0.444961,1,https://www.linkedin.com/jobs/view/3899512960/...


In [71]:
# --------------------------------------------------
# EVALUATION METRICS
# --------------------------------------------------

def is_relevant_title(job_title, relevant_titles):
    job_title = str(job_title).lower()

    for relevant_title in relevant_titles:
        if relevant_title.lower() in job_title:
            return True

    return False


def precision_at_k(recommendations_df, relevant_titles, k=10):
    recommended_titles = recommendations_df["title"].head(k).tolist()

    hits = sum(
        1 for title in recommended_titles
        if is_relevant_title(title, relevant_titles)
    )

    return hits / k


def recall_at_k(recommendations_df, relevant_titles, k=10):
    recommended_titles = recommendations_df["title"].head(k).tolist()

    hits = sum(
        1 for title in recommended_titles
        if is_relevant_title(title, relevant_titles)
    )

    if len(relevant_titles) == 0:
        return 0

    return hits / len(relevant_titles)

Matched skills = skills in both resume and job

Missing skills = skills required by job but missing from resume

In [72]:
#  create a skill dictionary 
skill_keywords = [
    "python", "java", "javascript", "sql", "excel", "tableau", "power bi",
    "machine learning", "deep learning", "nlp", "data analysis",
    "data visualization", "statistics", "pandas", "numpy", "scikit-learn",
    "tensorflow", "pytorch", "aws", "azure", "docker", "kubernetes",
    "react", "node.js", "html", "css", "git", "github",
    "communication", "leadership", "project management", "agile",
    "marketing", "seo", "sales", "recruitment", "hr", "finance"
]

In [73]:
#  skill extraction function
def extract_skills(text, skill_list):
    text = str(text).lower()
    found_skills = []

    for skill in skill_list:
        if skill.lower() in text:
            found_skills.append(skill)

    return sorted(list(set(found_skills)))

In [74]:
#  matched and msisng skill function
def get_skill_gap(resume_text, job_description):
    resume_skills = set(extract_skills(resume_text, skill_keywords))
    job_skills = set(extract_skills(job_description, skill_keywords))

    matched_skills = sorted(list(resume_skills.intersection(job_skills)))
    missing_skills = sorted(list(job_skills - resume_skills))

    return matched_skills, missing_skills

In [75]:
#add skill gap to recommendations
def recommend_jobs_with_skill_gap(resume_text, top_n=10):
    recommendations = recommend_jobs_hybrid(resume_text, top_n)

    matched_list = []
    missing_list = []

    for idx in recommendations.index:
        job_description = df.loc[idx, "description"]

        matched_skills, missing_skills = get_skill_gap(
            resume_text,
            job_description
        )

        matched_list.append(", ".join(matched_skills))
        missing_list.append(", ".join(missing_skills))

    recommendations["matched_skills"] = matched_list
    recommendations["missing_skills"] = missing_list

    return recommendations

In [76]:
# test
resume_text = """
Data analyst with experience in Python, SQL, Excel, Tableau, Power BI,
data visualization, dashboard development, reporting, statistics, and machine learning.
"""

recommend_jobs_with_skill_gap(resume_text, top_n=10)

,title,company_name,location,formatted_work_type,formatted_experience_level,normalized_salary,tfidf_score,sbert_score,hybrid_score,cluster,job_posting_url,matched_skills,missing_skills
2378,Business Analyst III - Tableau (Remote),Talentify.io,United States,Full-time,Mid-Senior level,150280.0,0.394115,0.701457,0.578520,1,https://www.linkedin.com/jobs/view/3905658764/...,"excel, power bi, python, sql, tableau","data analysis, hr"
1913,Power BI Developer,Odyssey Information Services,"Piqua, OH",Full-time,Mid-Senior level,-1.0,0.411496,0.582861,0.514315,1,https://www.linkedin.com/jobs/view/3887102526/...,"data visualization, excel, power bi, sql","communication, data analysis"
1834,Palantir Developer,Tata Consultancy Services,"Cleveland, OH",Full-time,Entry level,-1.0,0.495722,0.472806,0.481973,1,https://www.linkedin.com/jobs/view/3885815054/...,"data visualization, excel, machine learning, p...","azure, communication, data analysis, deep lear..."
5499,Power BI Developer,Optomi,United States,Contract,Mid-Senior level,133120.0,0.399939,0.527410,0.476422,1,https://www.linkedin.com/jobs/view/3887100812/...,"data visualization, excel, power bi, python, sql","agile, azure, communication, data analysis"
7986,Business Intelligence Analyst,Emergere Technologies LLC.,"Dallas, TX",Contract,Not Specified,-1.0,0.313178,0.580936,0.473833,1,https://www.linkedin.com/jobs/view/3900964322/...,"data visualization, excel, power bi, python","aws, communication, data analysis"
3740,Data Analyst - Power BI & SQL,ZealTech,"San Francisco, CA",Contract,Not Specified,-1.0,0.401656,0.518394,0.471699,1,https://www.linkedin.com/jobs/view/3885113214/...,"data visualization, excel, power bi, sql","communication, data analysis"
2376,Business Data Analyst,Catapult Solutions Group,"San Diego, CA",Full-time,Mid-Senior level,133712.8,0.440451,0.486560,0.468116,1,https://www.linkedin.com/jobs/view/3901389257/...,"data visualization, excel, machine learning, p...","communication, data analysis, hr, react"
3626,Data Science Intern DIN47,Data Glacier,United States,Internship,Not Specified,-1.0,0.296330,0.563501,0.456633,1,https://www.linkedin.com/jobs/view/3901939220/...,"data visualization, excel, machine learning, p...","communication, data analysis"
4160,Data Analyst,SNVA Technologies,"Juno Beach, FL",Contract,Not Specified,-1.0,0.274982,0.577639,0.456576,1,https://www.linkedin.com/jobs/view/3901373790/...,"excel, python, sql","data analysis, git"
2553,Data Analyst,Gravity IT Resources,"Juno Beach, FL",Contract,Mid-Senior level,72800.0,0.240047,0.574174,0.440523,1,https://www.linkedin.com/jobs/view/3905311477/...,"excel, power bi, python, sql","data analysis, git"


In [77]:
# Evaluattion
test_resumes = {
    "Data Analyst": """
    Data Analyst with experience in Python, SQL, Excel, Tableau,
    Power BI, dashboard reporting, data visualization, statistics,
    and machine learning.
    """,

    "Software Engineer": """
    Software Engineer skilled in Java, Python, REST APIs, Docker,
    AWS, Git, backend development, databases, and cloud deployment.
    """,

    "HR Recruiter": """
    HR professional with experience in recruitment, talent acquisition,
    interviewing, onboarding, employee relations, and communication.
    """,

    "Digital Marketing": """
    Digital marketer experienced in SEO, Google Analytics, social media,
    content marketing, campaign management, and sales strategy.
    """
}

In [78]:
def recommend_jobs_tfidf(resume_text, top_n=5):
    clean_resume = preprocess_text(resume_text)
    resume_vector = tfidf_vectorizer.transform([clean_resume])

    scores = cosine_similarity(resume_vector, tfidf_matrix).flatten()

    top_indices = scores.argsort()[::-1][:top_n]

    results = df.iloc[top_indices].copy()
    results["tfidf_score"] = scores[top_indices]

    return results[["title", "company_name", "location", "tfidf_score"]]

In [79]:
def recommend_jobs_sbert(resume_text, top_n=5):
    resume_embedding = sbert_model.encode([resume_text])
    resume_embedding_norm = normalize(resume_embedding)

    scores = cosine_similarity(
        resume_embedding_norm,
        job_embeddings_norm
    ).flatten()

    top_indices = scores.argsort()[::-1][:top_n]

    results = df.iloc[top_indices].copy()
    results["sbert_score"] = scores[top_indices]

    return results[["title", "company_name", "location", "sbert_score"]]

In [80]:
for resume_name, resume_text in test_resumes.items():
    print("\n" + "="*80)
    print(f"TEST RESUME: {resume_name}")
    print("="*80)

    print("\nTF-IDF Recommendations:")
    display(recommend_jobs_tfidf(resume_text, top_n=5))

    print("\nSBERT Recommendations:")
    display(recommend_jobs_sbert(resume_text, top_n=5))

    print("\nHybrid Recommendations:")
    display(recommend_jobs_hybrid(resume_text, top_n=5))


TEST RESUME: Data Analyst

TF-IDF Recommendations:


,title,company_name,location,tfidf_score
1834,Palantir Developer,Tata Consultancy Services,"Cleveland, OH",0.499038
2614,Lead Machine Learning Engineer,Confidential Jobs,United States,0.460960
2376,Business Data Analyst,Catapult Solutions Group,"San Diego, CA",0.443397
1913,Power BI Developer,Odyssey Information Services,"Piqua, OH",0.414248
3740,Data Analyst - Power BI & SQL,ZealTech,"San Francisco, CA",0.404343



SBERT Recommendations:


,title,company_name,location,sbert_score
2378,Business Analyst III - Tableau (Remote),Talentify.io,United States,0.686749
7986,Business Intelligence Analyst,Emergere Technologies LLC.,"Dallas, TX",0.572116
4160,Data Analyst,SNVA Technologies,"Juno Beach, FL",0.567607
1913,Power BI Developer,Odyssey Information Services,"Piqua, OH",0.566004
3626,Data Science Intern DIN47,Data Glacier,United States,0.563221



Hybrid Recommendations:


...

,title,company_name,location,tfidf_score
5613,Specialist,Funnel Boost Media,"San Antonio, TX",0.460571
6091,Article Writer,Offshore Consulting Services Group JM,"Massachusetts, United States",0.452272
166,Digital Marketing Specialist,PKB Pet King Brands: Makers of Zymox and Oratene,"Austin, Texas Metropolitan Area",0.443341
9843,Marketing Associate,PDI,"Woodcliff Lake, NJ",0.437720
3414,Marketing Strategist,Impulse Creative,United States,0.429632



SBERT Recommendations:


,title,company_name,location,sbert_score
5613,Specialist,Funnel Boost Media,"San Antonio, TX",0.605063
7803,Digital Marketing and Communication Intern,"iQuasar, LLC","Fredericksburg, VA",0.577151
9578,Marketo Developer,MSRcosmos LLC,"Warren, NJ",0.558741
1158,Marketing Assistant,SUCCESS Magazine,United States,0.548889
6270,Digital Ad Ops Coordinator,"Nexstar Media Group, Inc.","Bakersfield, CA",0.544059



Hybrid Recommendations:


,title,company_name,location,formatted_work_type,formatted_experience_level,normalized_salary,tfidf_score,sbert_score,hybrid_score,cluster,job_posting_url
5613,Specialist,Funnel Boost Media,"San Antonio, TX",Full-time,Not Specified,-1.0,0.460571,0.605063,0.547266,7,https://www.linkedin.com/jobs/view/3901977993/...
166,Digital Marketing Specialist,PKB Pet King Brands: Makers of Zymox and Oratene,"Austin, Texas Metropolitan Area",Full-time,Not Specified,46500.0,0.443341,0.534079,0.497784,7,https://www.linkedin.com/jobs/view/3901986081/...
7803,Digital Marketing and Communication Intern,"iQuasar, LLC","Fredericksburg, VA",Internship,Not Specified,-1.0,0.374055,0.577151,0.495913,7,https://www.linkedin.com/jobs/view/3902351941/...
6270,Digital Ad Ops Coordinator,"Nexstar Media Group, Inc.","Bakersfield, CA",Full-time,Entry level,-1.0,0.381663,0.544059,0.479100,7,https://www.linkedin.com/jobs/view/3903839921/...
1158,Marketing Assistant,SUCCESS Magazine,United States,Contract,Not Specified,52000.0,0.350297,0.548889,0.469452,7,https://www.linkedin.com/jobs/view/3902846521/...
